# 03 — Domain Classifier

**Phase 3 of the Domain-Adaptive Research Paper Summarization System**

Fine-tunes `bert-base-uncased` to classify papers as **General** (AI / CS / Physics)
or **Medical** (Biomedical / Clinical). The saved checkpoint becomes the routing
component in Phase 7 (Angle 2).

```
HuggingFace datasets (arXiv → General, PubMed → Medical)
       │  10 K samples per class
       ▼
Tokenize: title [SEP] abstract  →  bert-base-uncased (512 tokens)
       ▼
Fine-tune BertForSequenceClassification  (HuggingFace Trainer + wandb)
       ▼
Evaluate: accuracy · F1 · confusion matrix
       ▼
Save  →  outputs/classifier/
       ▼
Smoke-test  src/classifier.py  predict_domain()
```

> **HPC note:** This notebook is the interactive version. The equivalent
> batch job is `slurm/train_classifier.sbatch`. Run this notebook first
> to verify everything works, then submit the SLURM job for the full run
> if you want non-interactive execution.


## 0 — Configuration

In [2]:
# ── Mode ─────────────────────────────────────────────────────────────────────
DATASET_MODE = "full"   # "dev" | "full"  — full on HPC

# ── Imports ───────────────────────────────────────────────────────────────────
import json
import os
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import wandb
import yaml
from dotenv import load_dotenv
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from datasets import Dataset, DatasetDict
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

# ── Load environment (.env must contain WANDB_API_KEY and HF_DATASETS_CACHE) ─
load_dotenv()

# ── Config ────────────────────────────────────────────────────────────────────
with open("../configs/classifier_config.yaml") as f:
    cfg = yaml.safe_load(f)

SEED       = cfg["training"]["seed"]
MODEL_NAME = cfg["model"]["name"]
MAX_LEN    = cfg["model"]["max_input_tokens"]
NUM_LABELS = cfg["model"]["num_labels"]
LABEL_MAP  = {int(k): v for k, v in cfg["model"]["label_map"].items()}
OUTPUT_DIR = cfg["output"]["output_dir"]
LOG_DIR    = cfg["output"]["logging_dir"]
DATA_DIR   = Path(f"../data/processed_{DATASET_MODE}")

# ── Reproducibility ───────────────────────────────────────────────────────────
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"DATASET_MODE : {DATASET_MODE}")
print(f"Device       : {device}")
print(f"CUDA name    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"Model        : {MODEL_NAME}")
print(f"Label map    : {LABEL_MAP}")
print(f"WandB API key set: {'WANDB_API_KEY' in os.environ}")

DATASET_MODE : full
Device       : cpu
CUDA name    : N/A
Model        : bert-base-uncased
Label map    : {0: 'General', 1: 'Medical'}
WandB API key set: True


## 1 — Load Data

Full mode: sample 10 K papers per class from `ccdv/arxiv-summarization`
(General) and `ccdv/pubmed` (Medical) directly from HuggingFace.
Input text is `title [SEP] abstract` — short enough to fit in 512 tokens
while carrying a strong domain signal.

In [3]:
def load_json_split(path: Path) -> list[dict]:
    """Load a JSON list from disk (UTF-8 encoded).

    Args:
        path: Path to a .json file containing a list of record dicts.

    Returns:
        List of dicts.
    """
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def build_classifier_records_dev(records: list[dict]) -> list[dict]:
    """Convert processed dev records into (text, label) pairs.

    Args:
        records: Records from processed_dev JSON files.

    Returns:
        List of dicts with keys ``"text"`` and ``"label"``.
    """
    label_to_id = {v: k for k, v in LABEL_MAP.items()}
    out = []
    for r in records:
        title   = (r.get("title")   or "").strip()
        article = (r.get("article") or "").strip()
        domain  = r.get("domain", "General")
        text_in = title + " [SEP] " + article[:1000]
        label   = label_to_id.get(domain, 0)
        out.append({"text": text_in, "label": label})
    return out


if DATASET_MODE == "dev":
    train_raw = load_json_split(DATA_DIR / "train.json")
    val_raw   = load_json_split(DATA_DIR / "validation.json")
    test_raw  = load_json_split(DATA_DIR / "test.json")
    train_cls = build_classifier_records_dev(train_raw)
    val_cls   = build_classifier_records_dev(val_raw)
    test_cls  = build_classifier_records_dev(test_raw)

else:
    from datasets import load_dataset as hf_load

    samples_per_class = cfg["data"]["samples_per_class"]
    print(f"Sampling {samples_per_class} per class from HuggingFace…")

    arxiv_ds  = hf_load("ccdv/arxiv-summarization", split="train").shuffle(seed=SEED).select(range(samples_per_class))
    pubmed_ds = hf_load("ccdv/pubmed",              split="train").shuffle(seed=SEED).select(range(samples_per_class))

    def hf_to_cls(ds, label_id):
        """Convert a HuggingFace dataset split into (text, label) records.

        Args:
            ds:       HuggingFace Dataset split.
            label_id: Integer class label (0=General, 1=Medical).

        Returns:
            List of dicts with keys ``"text"`` and ``"label"``.
        """
        recs = []
        for row in ds:
            title    = (row.get("title")    or "").strip()
            abstract = (row.get("abstract") or "").strip()
            text_in  = title + " [SEP] " + abstract
            recs.append({"text": text_in, "label": label_id})
        return recs

    all_records = hf_to_cls(arxiv_ds, 0) + hf_to_cls(pubmed_ds, 1)
    random.shuffle(all_records)

    n         = len(all_records)
    n_train   = int(0.8 * n)
    n_val     = int(0.1 * n)
    train_cls = all_records[:n_train]
    val_cls   = all_records[n_train:n_train + n_val]
    test_cls  = all_records[n_train + n_val:]

print(f"Splits — train: {len(train_cls)}, val: {len(val_cls)}, test: {len(test_cls)}")
for split_name, split in [("train", train_cls), ("val", val_cls), ("test", test_cls)]:
    counts = pd.Series([r["label"] for r in split]).value_counts().to_dict()
    print(f"  {split_name}: { {LABEL_MAP[k]: v for k, v in counts.items()} }")

Sampling 10000 per class from HuggingFace…


Using the latest cached version of the dataset since ccdv/arxiv-summarization couldn't be found on the Hugging Face Hub


ValueError: There are multiple 'ccdv/arxiv-summarization' configurations in the cache: default, section
Please specify which configuration to reload from the cache, e.g.
	load_dataset('ccdv/arxiv-summarization', 'default')

## 2 — Tokenize

In [ ]:
hf_ds = DatasetDict({
    "train":      Dataset.from_list(train_cls),
    "validation": Dataset.from_list(val_cls),
    "test":       Dataset.from_list(test_cls),
})

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    """Tokenize a batch of classifier input texts.

    Args:
        batch: Dict with key ``"text"`` (list of strings).

    Returns:
        Dict of input_ids, attention_mask, token_type_ids.
    """
    return tokenizer(
        batch["text"],
        max_length=MAX_LEN,
        truncation=True,
        padding="max_length",
    )

tokenized = hf_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
tokenized.set_format("torch")

print("Columns :", tokenized["train"].column_names)
print("input_ids shape:", tokenized["train"][0]["input_ids"].shape)

## 3 — Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS
)
model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params     : {total_params:,}")
print(f"Trainable params : {trainable_params:,}")

## 4 — Metrics

In [ ]:
def compute_metrics(eval_pred):
    """Compute accuracy and weighted F1 from Trainer eval predictions.

    Args:
        eval_pred: EvalPrediction namedtuple (predictions, label_ids).

    Returns:
        Dict with keys ``"accuracy"`` and ``"f1"``.
    """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1":       f1_score(labels, preds, average="weighted"),
    }

## 5 — Training Arguments

In [ ]:
t = cfg["training"]

training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    logging_dir                 = LOG_DIR,
    num_train_epochs            = t["epochs"],
    per_device_train_batch_size = t["per_device_train_batch_size"],
    per_device_eval_batch_size  = t["per_device_eval_batch_size"],
    gradient_accumulation_steps = t["gradient_accumulation_steps"],
    optim                       = t["optimizer"],
    learning_rate               = t["learning_rate"],
    lr_scheduler_type           = t["lr_scheduler_type"],
    warmup_ratio                = t["warmup_ratio"],
    weight_decay                = t["weight_decay"],
    fp16                        = True,   # GPU always available on HPC
    evaluation_strategy         = t["evaluation_strategy"],
    save_strategy               = t["save_strategy"],
    load_best_model_at_end      = t["load_best_model_at_end"],
    metric_for_best_model       = t["metric_for_best_model"],
    greater_is_better           = True,
    seed                        = SEED,
    report_to                   = "wandb",
    run_name                    = cfg["wandb"]["run_name"],
    logging_steps               = 50,
)

print(f"Epochs         : {t['epochs']}")
print(f"Effective batch: {t['per_device_train_batch_size'] * t['gradient_accumulation_steps']}")
print(f"fp16           : {training_args.fp16}")
print(f"Report to      : {training_args.report_to}")

## 6 — Fine-tune

In [ ]:
wandb.init(
    project = cfg["wandb"]["project"],
    name    = cfg["wandb"]["run_name"],
    config  = cfg,
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = tokenized["train"],
    eval_dataset    = tokenized["validation"],
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Starting training…")
train_result = trainer.train()
print("Training complete.")
print(train_result.metrics)

## 7 — Training Curves

In [ ]:
log_history  = trainer.state.log_history

train_steps  = [e["step"]      for e in log_history if "loss"      in e]
train_losses = [e["loss"]      for e in log_history if "loss"      in e]
eval_steps   = [e["step"]      for e in log_history if "eval_loss" in e]
eval_losses  = [e["eval_loss"] for e in log_history if "eval_loss" in e]
eval_f1s     = [e.get("eval_f1") for e in log_history if "eval_loss" in e]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_steps, train_losses, label="Train loss",      color="steelblue")
axes[0].plot(eval_steps,  eval_losses,  label="Validation loss", color="tomato", marker="o")
axes[0].set_title("Loss curves")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].legend()

if any(v is not None for v in eval_f1s):
    axes[1].plot(eval_steps, eval_f1s, label="Validation F1", color="seagreen", marker="o")
    axes[1].set_title("Validation F1")
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Weighted F1")
    axes[1].legend()

plt.tight_layout()
Path("results").mkdir(exist_ok=True)
plt.savefig("results/classifier_training_curves.png", dpi=150)
plt.show()
print("Saved → results/classifier_training_curves.png")

## 8 — Test Set Evaluation

In [ ]:
test_output = trainer.predict(tokenized["test"])
preds       = np.argmax(test_output.predictions, axis=-1)
labels      = test_output.label_ids

acc    = accuracy_score(labels, preds)
f1_w   = f1_score(labels, preds, average="weighted")
report = classification_report(labels, preds, target_names=[LABEL_MAP[0], LABEL_MAP[1]])
cm     = confusion_matrix(labels, preds)

print(f"Test Accuracy       : {acc:.4f}")
print(f"Test F1 (weighted)  : {f1_w:.4f}")
print("\nClassification Report:")
print(report)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[LABEL_MAP[0], LABEL_MAP[1]]
).plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix — full mode")
plt.tight_layout()
plt.savefig("results/classifier_confusion_matrix.png", dpi=150)
plt.show()
print("Saved → results/classifier_confusion_matrix.png")

Path("results").mkdir(exist_ok=True)
with open("results/classifier_test_metrics.json", "w", encoding="utf-8") as f:
    json.dump({
        "dataset_mode"     : DATASET_MODE,
        "test_accuracy"    : round(acc,  4),
        "test_f1_weighted" : round(f1_w, 4),
        "confusion_matrix" : cm.tolist(),
    }, f, indent=2)
print("Saved → results/classifier_test_metrics.json")

## 9 — Save Checkpoint

In [ ]:
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Checkpoint saved → {OUTPUT_DIR}")
print("Files:", [p.name for p in Path(OUTPUT_DIR).iterdir()])

wandb.finish()

## 10 — Smoke-test `src/classifier.py`

In [ ]:
sys.path.insert(0, str(Path(".").resolve()))
from src.classifier import load_classifier, predict_domain, batch_predict

clf_pipeline = load_classifier(OUTPUT_DIR)
print(f"Classifier loaded on: {clf_pipeline.device}")

test_texts = [
    (
        "Attention Is All You Need [SEP] We propose the Transformer, a model "
        "architecture based solely on attention mechanisms, dispensing with recurrence.",
        "Expected: General",
    ),
    (
        "SARS-CoV-2 mRNA Vaccine Efficacy [SEP] A randomised double-blind "
        "placebo-controlled Phase III trial showed 95% efficacy against COVID-19.",
        "Expected: Medical",
    ),
    (
        "Graph Neural Networks for Drug Discovery [SEP] We apply GNNs to predict "
        "ADMET properties of candidate molecules on the MoleculeNet benchmark.",
        "Expected: ambiguous",
    ),
]

print("\n=== predict_domain() ===")
for text, note in test_texts:
    label, conf = predict_domain(text, clf_pipeline)
    print(f"  {label:8s}  conf={conf:.4f}  [{note}]")

print("\n=== batch_predict() ===")
for res, (_, note) in zip(batch_predict([t for t, _ in test_texts], clf_pipeline), test_texts):
    print(f"  {res['label']:8s}  conf={res['confidence']:.4f}  [{note}]")

## Summary

| Item | Status |
|---|---|
| 20 K classifier examples (10 K per class) | ✅ |
| Tokenized with bert-base-uncased | ✅ |
| Fine-tuned 3 epochs, logged to wandb | ✅ |
| Training curves saved | ✅ |
| Test accuracy + F1 + confusion matrix saved | ✅ |
| Checkpoint → `outputs/classifier/` | ✅ |
| `src/classifier.py` smoke-tested | ✅ |

**Next → Phase 4:** Submit `slurm/train_led_arxiv.sbatch` and open
`04_led_finetuning.ipynb` to fine-tune LED on arXiv.
